<a href="https://colab.research.google.com/github/dgromann/cl_intro_ws2026/blob/main/exercises/HomeExercise2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Home Exericse 2: Word Embeddings
In this second home exercise, you will use the knowledge from Tutorial 4 to perform a more systematic evaluation of embeddings based on a small analogy dataset.

In this notebook, please complete all instructions starting with 👋 ⚒ in the code cell after the sign or provide your analysis in the text cell after the sign.

---

## **Part 1: Implementing a tiny Skip-gram in PyTorch**

The objective of Skipgram is to assign a high score to true neighbors of a center word and low scores to random words called negative samples. The paremeters of the embedding models are two embedding matrices, one that connects the input layer to the hidden layer (representing center words) and one that connects the hidden layer to the output layer (representing context words). Training is performed the exact same way as in Tutorial 4.

We need to install gensim so that we can load a corpus for training. Run the following code cell and then restart your runtime environment by clicking on Runtime in the menu and selecting Restart Runtime. This is necessary because of the downversioning of smart_open for gensim.

In [ ]:
!pip install gensim
#Due to incompatibility between gensim smart_open and Colab pre-installed smart_open
!pip install "smart_open<7"

We will use a cleaned English Wikipedia corpus of approximately 30 MB for this task. We will first get 1,000,000 tokens from the corpus and then calculate the 5,000 most frequent types that represents our vocabulary.

We need to create an index that maps each unique word (type) in the vocabulary to a unique number called `word2id` in the following code cell. Then, we assign each word in the corpus the corresponding id so that we can input one sentence after another into the model for training.

In [ ]:
import itertools
import gensim.downloader as api
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import Counter


corpus = api.load("text8")
tokens = list(itertools.islice(itertools.chain.from_iterable(corpus), 1_000_000))
print(len(tokens), "tokens")

V = 5000
vocab = [w for w, _ in Counter(tokens).most_common(V)]
word2id = {w: i for i, w in enumerate(vocab)}
ids = torch.tensor([word2id.get(w, -1) for w in tokens]) # -1 represents unknown words, that is, words not in the vocabulary

print(f"Creating an index/lookup dictionary that maps words to ids: {word2id}")
print(f"Mapping all 1000000 tokens in the text to ids: {len(ids)}")

Next we need to create our true neighbor pairs and our negative samples for training.

In [ ]:
# (center, context) pairs are created by looking at d positions to the right and adding mirror pairs
window_size = 4
centers, contexts = [], []
for d in range(1, window_size + 1):
    a, b = ids[:-d], ids[d:]
    ok = (a >= 0) & (b >= 0)
    centers += [a[ok], b[ok]]
    contexts += [b[ok], a[ok]]
centers, contexts = torch.cat(centers), torch.cat(contexts)
print(len(centers), "training pairs")

# Negative distances are calculated as the probability of a specific word to be a negative sample with sample words by frequency^0.75 (as in word2vec)
counts = torch.bincount(ids[ids >= 0], minlength=V).float()
neg_dist = counts ** 0.75
neg_dist /= neg_dist.sum()
print(len(neg_dist), "each word gets a probability of picking that word as a negative, the more frequent the higher the probability")

As a next step we need to compose the model with all its layers and the training step before we can actually train. We will keep it very small with only a dimensionality of 50. We have two embedding layers, one connecting the input layer to the hidden layer (center.weight) and the other connecting the hidden layer to the output layer (context.weight). Then we have one forward pass, where we need to obtain the vector for the center word `vc`, the vector for the context word `uo`, and the vector for the negative sample `un`. For every true pair, we draw K numbers of negative samples.

👋 ⚒ Exercise 1: Complete the forward pass of the following SkipGram model by adding the correct calculation of the true pair, which is the multiplication of two vectors that represent the true pair and then the sum over dim=1.

In [102]:
class SkipGram(nn.Module):
    def __init__(self, vocab_size, dim=50):
        super().__init__()
        self.center  = nn.Embedding(vocab_size, dim)
        self.context = nn.Embedding(vocab_size, dim)
        nn.init.uniform_(self.center.weight,  -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.context.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, c, o, neg):
        # c: (B,) centre ids   o: (B,) true context ids   neg: (B, K) negative ids
        vc = self.center(c)           # (B, dim)
        uo = self.context(o)          # (B, dim)
        un = self.context(neg)        # (B, K, dim)


        pos_score =                  # Your code here: complete the pos_score calculation
        neg_score = torch.bmm(un, vc.unsqueeze(2)).squeeze(2)   # (B, K), given

        return -(F.logsigmoid(pos_score) + F.logsigmoid(-neg_score).sum(dim=1)).mean()

After having defined the model, all the parameters (weights) in the model are random, which means it cannot identify words that are supposed to be neighbors. Run the following code cell to have a look at the performance before training.

In [ ]:
def neighbors(word, emb_layer, k=8):
  emb_layer = emb_layer - emb_layer.mean(dim=0)                 # remove the common direction
  embeddings = emb_layer / emb_layer.norm(dim=1, keepdim=True)  # unit-length rows
  sims = embeddings @ embeddings[word2id[word]]                 # calculate cosine with every word
  top = sims.topk(k + 1).indices[1:].tolist()                   # keep only the top k words and return them
  return [vocab[t] for t in top]

model = SkipGram(V)
# The loaded corpus was already in lowercase
for w_ in ["king", "france", "music"]:
  print(w_, neighbors(w_, model.center.weight.detach()))

Then we need to perform the actual training step.

👋 ⚒ Exercise 2: Complete the loss and optimization steps in the following training loop.

In [ ]:
model = SkipGram(V)
opt = torch.optim.Adam(model.parameters(), lr=0.003)
# K is the number of negative samples we draw from the vocabulary (bag of words)
# Batch size is the number of (center, context) word pairs processed in one training and weight update step
K, batch_size = 5, 8192
history = []

for epoch in range(3):
    perm = torch.randperm(len(centers))
    total, n = 0.0, 0
    losses = []
    for s in range(0, len(perm), batch_size):
        idx = perm[s:s + batch_size]
        c, o = centers[idx], contexts[idx]
        neg = torch.multinomial(neg_dist, len(idx) * K, replacement=True).view(len(idx), K)

        # Your code here for loss and optimization
        # To calculate the loss you need to perform a forward pass by using model() and
        # handing over the right variables for a forward pass


        losses.append(loss.item())

        total += loss.item(); n += 1

    history.append(total / n)
    print(f"Epoch {epoch}: Loss {history[-1]:.4f}")
    print(f"To take a look at all recorded loss per epoch, you can print losses: {losses}")

---

## **Part 2: Evaluating your Skipgram model**

👋 ⚒ Exercise 3: Use the neighbors function from the previous part to test which words are the nearest neighbors of the following words. Do the provided nearest neighbors have a closer relation to the word than before training?



In [ ]:
words = ["king", "france", "music", "water", "computer"]
# Your code

👋 ⚒ Exercise 4: Add another code cell here were you continue training the same model for another 3 epochs. You need to copy most of the code from the end of Part 1 but not all of it.

Mind that the loss of Epoch 0 should be lower than the previous loss of Epoch 2 in Part 1, since you continue the training from that last epoch.

In [ ]:
# Your code here

👋 ⚒ Exercise 5: Evaluate your own embedding model that has now been trained for six instead of three epochs  on the nearest neighbors of our word example list. Have the nearest neighbors further improved?

In [ ]:
# Your code here

Load the already trained GloVe model for the next excerise.

In [85]:
glove_embeddings = api.load("glove-wiki-gigaword-50")


👋 ⚒ Exercise 6: Compare these results to the nearest neighbors of `words` in the already trained GloVe model from Tutorial 4. Which embedding model has better neighbors in your view? The tiny Skip-gram or the GloVe model?

In [ ]:
# Your code

👋 ⚒ Exercise 7: Use the analogy calculation method to evaluate potential biases in embedding models. One example of a gender bias in professions is already provided. Come up and test three more examples of bias, e.g. related to culture, gender, race, ethnicity, etc.

This shows that when training on text only, the model automatically learns bias inherent in the texts.

In [ ]:
bias_analysis1 = glove_embeddings.most_similar(positive=["doctor", "woman"], negative=["man"], topn=3)
print(f"man is to doctor as woman is to {bias_analysis1}")

# Your code and bias examples here